- Bước 1: Load video lên  
- Bước 2: Tách theo từng frame và duyệt theo từng frame  
    - Bước 2.1: Chạy detect trên frame i  
    - Bước 2.2: Cắt từng box output của frame i  
        - Bước 2.2.1: Chạy extract feature trên từng box j của frame i  
        - Bước 2.2.2: Lấy feature vector của các box j của frame i  
    - Bước 2.3: Chạy tracker kết nối frame i với các frame i-1, i-2 bằng kết quả detect/appearance feature tương ứng với mỗi loại tracker.  
- Bước 3: Offline refinement với GTALink, trả về danh sách tracklet id, frame và vị trí tại mỗi frame.
- Bước 4: Visualize.

In [1]:
import cv2
import os
import numpy as np
from typing import List, Dict, Optional
from src.feature_extractor import FeatureExtractor
from src.detector import Detector
from src.tracker import Tracker
from src.gtalink import GTALink
from src.compute_metrics import compute_metrics

# Path

In [2]:
#BASE = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT"
BASE = r"D:\UIT 3rd year\NhanDang\Project"

YOLO_PATH = os.path.join(BASE, r"PipelineMOT\models\best_yolo26.pt")
RF_PATH = os.path.join(BASE, r"PipelineMOT\models\best_rf_detr.pth")
SOLIDER_PATH = os.path.join(BASE, r"PipelineMOT\models\swin_small_converted.pth")
VIDEO_PATH = os.path.join(BASE, r"Data\wide_view\videos\F_20220220_1_1800_1830.mp4")
GT_CSV_PATH = os.path.join(BASE, r"Data\wide_view\annotations\F_20220220_1_1800_1830.csv")
OUTPUT_DIR = os.path.join(BASE, r"Output")

os.makedirs(OUTPUT_DIR, exist_ok=True)

ALL_DETECTOR  = ["yolo26", "rf_detr"]
ALL_EXTRACTOR = ["osnet", "solider", "color_histogram"]
ALL_TRACKER   = ["bytetrack", "ocsort", "strongsort", "deepeiou"]

# PIPELINE FUNCTIONS

In [3]:
_PALETTE = [
    (255, 99, 71),    # Tomato
    (255, 165, 0),    # Orange
    (255, 215, 0),    # Gold
    (154, 205, 50),   # YellowGreen
    (50, 205, 50),    # LimeGreen
    (0, 255, 127),    # SpringGreen
    (0, 206, 209),    # DarkTurquoise
    (30, 144, 255),   # DodgerBlue
    (65, 105, 225),   # RoyalBlue
    (138, 43, 226),   # BlueViolet
    (186, 85, 211),   # MediumOrchid
    (255, 20, 147),   # DeepPink
    (255, 105, 180),  # HotPink
    (210, 180, 140),  # Tan
    (244, 164, 96),   # SandyBrown
    (46, 139, 87),    # SeaGreen
    (70, 130, 180),   # SteelBlue
    (123, 104, 238),  # MediumSlateBlue
    (199, 21, 133),   # MediumVioletRed
    (255, 69, 0),     # OrangeRed
    (0, 191, 255),    # DeepSkyBlue
    (127, 255, 212),  # Aquamarine
]

def _color(track_id: int):
    return _PALETTE[track_id % len(_PALETTE)]

In [4]:
def run_mot(video_path, detector, tracker, extractor=None, refiner = None, extractor_refine = None):
    """
    Chạy full MOT pipeline cho một video.
 
    Parameters
    ----------
    video_path : str
    detector   : object với method .detect(frame) → [[x1,y1,x2,y2,conf], ...]
    tracker    : Tracker instance
    extractor  : object với method .extract(crop) → np.ndarray (L2-normalised), dùng cho online tracking
    refiner    : (optional) object với .refine(all_tracks) → refined_tracks
                 Nếu None, trả về all_tracks trực tiếp.
    extractor_refine: (optional) object với method .extract(crop) → np.ndarray
                       Extractor riêng để tạo embedding cho refinement.
                       Nếu None nhưng refiner được truyền vào → dùng chung với online tracking
 
    Returns
    -------
    all_tracks : dict
        {
            track_id (int): {
                'boxes' : list — box [x1,y1,x2,y2] hoặc None theo từng frame,
                'frames': list — frame_idx tương ứng (chỉ các frame có box)
            }
        }
    """
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Không mở được video: {video_path}")
    
    # all_tracks[track_id]['boxes'][i] = box tại frame i, hoặc None nếu không có
    all_tracks: Dict[int, Dict] = {} 

    frame_idx = 0   # 0-based index để index vào list
    frame_id  = 1   # 1-based id truyền vào tracker (convention của STrack)
    
    # extractor của refiner
    extractor_refine = extractor_refine if (refiner is not None and extractor_refine is not None) else extractor
    
    # Duyệt qua từng frame 
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        print(f"Processing frame {frame_idx}")
        
        # Bước 2.1: Detect
        detections = detector.detect(frame) # Trả về list các box và conf [[x1, y1, x2, y2, conf]]
        
        # Bước 2.2: Crop + Feature
        enriched_detections = [] # Tổng hợp box, conf, feature tại frame đang xét của các object
        for det in detections:
            enriched = {}
            x1, y1, x2, y2 = map(int, det[:-1])
            score = float(det[4])
            
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # 2.2.1 Extract feature
            feature = extractor.extract(crop) if extractor is not None else None

            # 2.2.2 attach feature
            enriched_detections.append({
                'tlbr' : [float(x1), float(y1), float(x2), float(y2)],
                'score': score,
                'feat' : feature,
            })

        # Bước 2.3: Tracking
        active_tracks  = tracker.update(enriched_detections, frame_id) 
         
        # Lưu all_tracks
        # Với các track_id mới: chèn None cho tất cả frame trước đó
        active_ids = set()
        for track in active_tracks:
            tid = track.track_id
            active_ids.add(tid)
 
            if tid not in all_tracks:
                # Track mới → fill None cho các frame trước
                entry = {
                    'boxes' : [None] * frame_idx,
                    'frames': [],
                }
                if refiner is not None:
                    entry['feats'] = []         # chỉ khởi tạo khi cần refinement
                all_tracks[tid] = entry
 
            box = track.tlbr.tolist()   # [x1,y1,x2,y2] 
            all_tracks[tid]['boxes'].append(box)
            all_tracks[tid]['frames'].append(frame_idx)
 
            # Nếu cần refine:
            if refiner is not None:
                    x1, y1, x2, y2 = map(int, track.tlbr)
                    # Clamp để tránh ra ngoài biên frame
                    crop = frame[y1:y2, x1:x2]
                    feat = extractor_refine.extract(crop) if crop.size > 0 \
                        else extractor_refine.extract(frame[0:1, 0:1])  # fallback crop rỗng
                    all_tracks[tid]['feats'].append(feat)
                    
        # Track đã có trong dict nhưng không active frame này → None
        for tid, data in all_tracks.items():
            if tid not in active_ids:
                # Chỉ thêm None nếu track chưa có entry cho frame này
                if len(data['boxes']) == frame_idx:
                    data['boxes'].append(None)
 
        frame_idx += 1
        frame_id  += 1
        
    cap.release()
    print(f"\n[MOT] Done — {frame_idx} frames, {len(all_tracks)} tracks")

    # Bước 3: Offline refinement
    if refiner is not None:
        return refiner.refine(all_tracks)   #all_tracks có feat nhưng bỏ feat sau output refiner
    return all_tracks

In [5]:
def visualize_tracks(video_path: str,
                     all_tracks: Dict,
                     output_path: str = "output_tracked.avi",  # ← Dùng .avi
                     show_id: bool = True,
                     thickness: int = 2,
                     font_scale: float = 0.7):
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Không mở được video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Buộc dùng AVI + MJPG (tương thích cao nhất với Windows)
    if not output_path.lower().endswith('.avi'):
        output_path = output_path.rsplit('.', 1)[0] + '.avi'

    fourcc = cv2.VideoWriter_fourcc(*'MJPG')      # Codec dễ mở nhất
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not writer.isOpened():
        print("⚠️ MJPG không hoạt động, thử XVID...")
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Xây frame map
    frame_map: Dict[int, Dict[int, List]] = {}
    for tid, data in all_tracks.items():
        for fi, box in enumerate(data.get('boxes', [])):
            if box is not None:
                frame_map.setdefault(fi, {})[tid] = box

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        for tid, box in frame_map.get(frame_idx, {}).items():
            x1, y1, x2, y2 = map(int, box)
            color = _color(tid)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

            if show_id:
                label = f"ID {tid}"
                (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
                label_y = max(y1 - 5, th + 5)

                cv2.rectangle(frame, 
                              (x1, label_y - th - baseline - 4),
                              (x1 + tw + 6, label_y + 2), 
                              color, cv2.FILLED)
                
                cv2.putText(frame, label, (x1 + 3, label_y - baseline),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255,255,255), thickness, cv2.LINE_AA)

        cv2.putText(frame, f"Frame {frame_idx+1}/{total}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (220,220,220), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"✅ Đã lưu xong: {output_path}")
    print("   → Hãy thử mở file này bằng Windows Media Player")

In [6]:
def run_and_visualize(video_path: str,
                      detector,
                      extractor,
                      output_path: str = 'output_tracked.avi',
                      algorithm: str = 'deepeiou',
                      refiner=None,
                      refiner_extractor = None,
                      tracker_kwargs: Optional[Dict] = None,
                      **viz_kwargs) -> Dict:
    
    tracker_kwargs = tracker_kwargs or {}
    tracker = Tracker(algorithm=algorithm, **tracker_kwargs)
 
    all_tracks = run_mot(
        video_path=video_path,
        detector=detector,
        tracker=tracker,
        extractor=extractor,
        refiner=refiner,
        extractor_refine=refiner_extractor
    )
 
    visualize_tracks(
        video_path=video_path,
        all_tracks=all_tracks,
        output_path=output_path,
        **viz_kwargs,
    )
 
    return all_tracks   # lấy về để tính theo metrics so sánh

In [ ]:
def run_pipeline(
    video_path:        str,
    gt_csv_path:       str,
    detector,
    tracker_name:      str,
    output_path:       str,
    extractor=None,                # online extractor (DeepEIoU dùng, ByteTrack bỏ qua)
    refiner=None,
    refiner_extractor=None,        # extractor cho GTALink
    iou_thresh:        float = 0.5,
    tracker_kwargs:    Optional[Dict] = None,
    **viz_kwargs
) -> Dict:
    """
    Chạy full pipeline MOT và tính metrics.

    Parameters
    ----------
    video_path        : đường dẫn video
    gt_csv_path       : đường dẫn file annotation CSV (để tính metrics)
    detector          : Detector instance
    tracker_name      : tên tracker ("deepeiou" | "bytetrack" | "ocsort" | "strongsort")
    output_path       : đường dẫn lưu video output
    extractor         : extractor cho online tracking (None nếu tracker không dùng)
    refiner           : GTALink instance (None nếu không dùng refinement)
    refiner_extractor : extractor cho GTALink (None nếu không dùng refinement)
    iou_thresh        : IoU threshold để tính metrics
    tracker_kwargs    : kwargs truyền vào tracker constructor

    Returns
    -------
    dict với all_tracks và metrics
    """
    if tracker_name in ('strongsort', 'deepeiou') and extractor is None:
        raise ValueError(
            f"Tracker '{tracker_name}' yêu cầu appearance extractor. "
            f"Vui lòng truyền extractor != None."
        )
    tracker_kwargs = tracker_kwargs or {}
    tracker        = Tracker(algorithm=tracker_name, **tracker_kwargs)
    ext_name = extractor.backend if extractor is not None else 'None'
    ref_name = refiner_extractor.backend if refiner_extractor is not None else \
               (extractor.backend if extractor is not None else 'None')
 
    print(f"\n{'='*60}")
    print(f"PIPELINE: {detector.backend.upper()} → "
          f"{ext_name.upper()} → "
          f"{tracker_name.upper()} → "
          f"GTALink({ref_name})")
    print(f"{'='*60}")

    # Bước 1-3: MOT
    all_tracks = run_mot(
        video_path      = video_path,
        detector        = detector,
        tracker         = tracker,
        extractor       = extractor,
        refiner         = refiner,
        extractor_refine= refiner_extractor,
    )

    # Bước 4: Visualize
    visualize_tracks(
        video_path  = video_path,
        all_tracks  = all_tracks,
        output_path = output_path,
        **viz_kwargs,
    )

    # Tính metrics
    metrics = compute_metrics(
        all_tracks = all_tracks,
        csv_path   = gt_csv_path,
        iou_thresh = iou_thresh,
    )

    return {'all_tracks': all_tracks, 'metrics': metrics}

# KHỞI TẠO COMPONENTS

In [17]:
detector          = Detector(backend="yolo26", model_path=YOLO_PATH)
extractor_color   = FeatureExtractor(backend="color_histogram")
extractor_osnet   = FeatureExtractor(backend="osnet")
extractor_solider = FeatureExtractor(
    backend="solider",
    solider_model_path=SOLIDER_PATH,
    solider_arch="swin_small",
    solider_semantic_weight=0.2,
)

[Detector] YOLO loaded on cuda
Successfully loaded imagenet pretrained weights from "C:\Users\Admin/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
[FeatureExtractor] OSNet loaded on cuda
Missing: 0 | Unexpected: 8
[FeatureExtractor] SOLIDER (swin_small) loaded on cuda
[FeatureExtractor] Feature dim: 768


# BYTETRACK (12 pipelines)

### P01: ByteTrack | None | GTALink(Color)

In [18]:
result_1 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p01_yolo_none_bytetrack_gtalink(color).avi"),
    extractor=None, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"lambda_iou": 1.0},
)


PIPELINE: YOLO26 → NONE → BYTETRACK → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proces

Splitting tracklets: 100%|██████████| 66/66 [00:00<00:00, 150.34it/s]


[GTALink] Sau  split: 87 tracklets
[GTALink] Trước merge: 87 tracklets
[GTALink] Sau  merge: 47 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p01_yolo_none_bytetrack_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15026 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    72.78 %
  MOTP  :    65.40 %
  IDF1  :    68.84 %
  HOTA  :    29.72 %
  IDs   :      131
  FP    :     1443
  FN    :     2917
  GT    :    16500


### P02: ByteTrack | None | GTALink(OSNet)

In [19]:
result_2 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p02_yolo_none_bytetrack_gtalink(osnet).avi"),
    extractor=None, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"lambda_iou": 1.0},
)


PIPELINE: YOLO26 → NONE → BYTETRACK → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame

Splitting tracklets: 100%|██████████| 66/66 [00:00<00:00, 231.97it/s]


[GTALink] Sau  split: 83 tracklets
[GTALink] Trước merge: 83 tracklets
[GTALink] Sau  merge: 26 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p02_yolo_none_bytetrack_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15026 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.24 %
  MOTP  :    65.40 %
  IDF1  :    73.77 %
  HOTA  :    36.85 %
  IDs   :       55
  FP    :     1443
  FN    :     2917
  GT    :    16500


### P03: ByteTrack | None | GTALink(Solider)

In [20]:
result_3 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p03_yolo_none_bytetrack_gtalink(solider).avi"),
    extractor=None, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"lambda_iou": 1.0},
)


PIPELINE: YOLO26 → NONE → BYTETRACK → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fra

Splitting tracklets: 100%|██████████| 66/66 [00:00<00:00, 198.26it/s]


[GTALink] Sau  split: 74 tracklets
[GTALink] Trước merge: 74 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p03_yolo_none_bytetrack_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15026 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.54 %
  MOTP  :    65.40 %
  IDF1  :    71.27 %
  HOTA  :    34.07 %
  IDs   :        6
  FP    :     1443
  FN    :     2917
  GT    :    16500


### P04: ByteTrack | Color | GTALink(Color)

In [21]:
result_4 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p04_yolo_color_bytetrack_gtalink(color).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → BYTETRACK → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing fram

Splitting tracklets: 100%|██████████| 66/66 [00:00<00:00, 212.01it/s]


[GTALink] Sau  split: 87 tracklets
[GTALink] Trước merge: 87 tracklets
[GTALink] Sau  merge: 48 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p04_yolo_color_bytetrack_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14985 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    72.99 %
  MOTP  :    65.37 %
  IDF1  :    69.90 %
  HOTA  :    30.47 %
  IDs   :       68
  FP    :     1437
  FN    :     2952
  GT    :    16500


### P05: ByteTrack | Color | GTALink(OSNet)

In [22]:
result_5 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p05_yolo_color_bytetrack_gtalink(osnet).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → BYTETRACK → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proce

Splitting tracklets: 100%|██████████| 66/66 [00:00<00:00, 243.83it/s]


[GTALink] Sau  split: 84 tracklets
[GTALink] Trước merge: 84 tracklets
[GTALink] Sau  merge: 26 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p05_yolo_color_bytetrack_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14985 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    72.95 %
  MOTP  :    65.37 %
  IDF1  :    72.11 %
  HOTA  :    35.96 %
  IDs   :       75
  FP    :     1437
  FN    :     2952
  GT    :    16500


### P06: ByteTrack | Color | GTALink(Solider)

In [23]:
result_6 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p06_yolo_color_bytetrack_gtalink(solider).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → BYTETRACK → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Pro

Splitting tracklets: 100%|██████████| 66/66 [00:00<00:00, 205.42it/s]


[GTALink] Sau  split: 71 tracklets
[GTALink] Trước merge: 71 tracklets
[GTALink] Sau  merge: 25 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p06_yolo_color_bytetrack_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14985 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.39 %
  MOTP  :    65.37 %
  IDF1  :    73.00 %
  HOTA  :    36.19 %
  IDs   :        1
  FP    :     1437
  FN    :     2952
  GT    :    16500


### P07: ByteTrack | OSNet | GTALink(Color)

In [24]:
result_7 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p07_yolo_osnet_bytetrack_gtalink(color).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → OSNET → BYTETRACK → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proce

Splitting tracklets: 100%|██████████| 61/61 [00:00<00:00, 204.00it/s]


[GTALink] Sau  split: 81 tracklets
[GTALink] Trước merge: 81 tracklets
[GTALink] Sau  merge: 42 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p07_yolo_osnet_bytetrack_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15004 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.13 %
  MOTP  :    65.40 %
  IDF1  :    70.44 %
  HOTA  :    30.59 %
  IDs   :       68
  FP    :     1435
  FN    :     2931
  GT    :    16500


### P08: ByteTrack | OSNet | GTALink(OSNet)

In [25]:
result_8 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p08_yolo_osnet_bytetrack_gtalink(osnet).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → OSNET → BYTETRACK → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fram

Splitting tracklets: 100%|██████████| 61/61 [00:00<00:00, 219.75it/s]


[GTALink] Sau  split: 79 tracklets
[GTALink] Trước merge: 79 tracklets
[GTALink] Sau  merge: 25 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p08_yolo_osnet_bytetrack_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15004 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.08 %
  MOTP  :    65.40 %
  IDF1  :    72.92 %
  HOTA  :    34.55 %
  IDs   :       75
  FP    :     1435
  FN    :     2931
  GT    :    16500


### P09: ByteTrack | OSNet | GTALink(Solider)

In [26]:
result_9 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p09_yolo_osnet_bytetrack_gtalink(solider).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → OSNET → BYTETRACK → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fr

Splitting tracklets: 100%|██████████| 61/61 [00:00<00:00, 182.92it/s]


[GTALink] Sau  split: 67 tracklets
[GTALink] Trước merge: 67 tracklets
[GTALink] Sau  merge: 25 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p09_yolo_osnet_bytetrack_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15004 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.53 %
  MOTP  :    65.40 %
  IDF1  :    72.28 %
  HOTA  :    35.45 %
  IDs   :        1
  FP    :     1435
  FN    :     2931
  GT    :    16500


### P10: ByteTrack | Solider | GTALink(Color)

In [27]:
result_10 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p10_yolo_solider_bytetrack_gtalink(color).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → SOLIDER → BYTETRACK → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Pro

Splitting tracklets: 100%|██████████| 63/63 [00:00<00:00, 181.62it/s]


[GTALink] Sau  split: 84 tracklets
[GTALink] Trước merge: 84 tracklets
[GTALink] Sau  merge: 45 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p10_yolo_solider_bytetrack_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14992 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.08 %
  MOTP  :    65.39 %
  IDF1  :    69.71 %
  HOTA  :    30.42 %
  IDs   :       68
  FP    :     1433
  FN    :     2941
  GT    :    16500


### P11: ByteTrack | Solider | GTALink(OSNet)

In [28]:
result_11 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p11_yolo_solider_bytetrack_gtalink(osnet).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → SOLIDER → BYTETRACK → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fr

Splitting tracklets: 100%|██████████| 63/63 [00:00<00:00, 208.53it/s]


[GTALink] Sau  split: 81 tracklets
[GTALink] Trước merge: 81 tracklets
[GTALink] Sau  merge: 25 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p11_yolo_solider_bytetrack_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14992 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.04 %
  MOTP  :    65.39 %
  IDF1  :    71.91 %
  HOTA  :    34.03 %
  IDs   :       75
  FP    :     1433
  FN    :     2941
  GT    :    16500


### P12: ByteTrack | Solider | GTALink(Solider)

In [29]:
result_12 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "bytetrack",
    os.path.join(OUTPUT_DIR, "p12_yolo_solider_bytetrack_gtalink(solider).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"lambda_iou": 0.8},
)


PIPELINE: YOLO26 → SOLIDER → BYTETRACK → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing 

Splitting tracklets: 100%|██████████| 63/63 [00:00<00:00, 185.00it/s]


[GTALink] Sau  split: 69 tracklets
[GTALink] Trước merge: 69 tracklets
[GTALink] Sau  merge: 25 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p12_yolo_solider_bytetrack_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14992 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    73.48 %
  MOTP  :    65.39 %
  IDF1  :    72.84 %
  HOTA  :    35.05 %
  IDs   :        1
  FP    :     1433
  FN    :     2941
  GT    :    16500


# OCSORT (12 pipelines)

### Pipeline 13: YOLO26 + Color + StrongSORT + GTALink(Solider)

In [30]:
result_13 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p13_yolo_none_ocsort_gtalink(color).avi"),
    extractor=None, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"use_emb": False},
)


PIPELINE: YOLO26 → NONE → OCSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processin

Splitting tracklets: 100%|██████████| 135/135 [00:00<00:00, 336.46it/s]


[GTALink] Sau  split: 189 tracklets
[GTALink] Trước merge: 189 tracklets
[GTALink] Sau  merge: 77 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p13_yolo_none_ocsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14820 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    69.07 %
  MOTP  :    65.42 %
  IDF1  :    56.04 %
  HOTA  :    22.76 %
  IDs   :      300
  FP    :     1562
  FN    :     3242
  GT    :    16500


### P14: OCSSort | None | GTALink(OSNet)

In [31]:
result_14 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p14_yolo_none_ocsort_gtalink(osnet).avi"),
    extractor=None, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"use_emb": False},
)


PIPELINE: YOLO26 → NONE → OCSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame 47

Splitting tracklets: 100%|██████████| 135/135 [00:00<00:00, 256.06it/s]


[GTALink] Sau  split: 200 tracklets
[GTALink] Trước merge: 200 tracklets
[GTALink] Sau  merge: 35 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p14_yolo_none_ocsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14820 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    68.66 %
  MOTP  :    65.42 %
  IDF1  :    65.52 %
  HOTA  :    27.02 %
  IDs   :      367
  FP    :     1562
  FN    :     3242
  GT    :    16500


### P15: OCSSort | None | GTALink(Solider)

In [32]:
result_15 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p15_yolo_none_ocsort_gtalink(solider).avi"),
    extractor=None, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"use_emb": False},
)


PIPELINE: YOLO26 → NONE → OCSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame 

Splitting tracklets: 100%|██████████| 135/135 [00:00<00:00, 341.34it/s]


[GTALink] Sau  split: 156 tracklets
[GTALink] Trước merge: 156 tracklets
[GTALink] Sau  merge: 33 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p15_yolo_none_ocsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14820 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    70.73 %
  MOTP  :    65.42 %
  IDF1  :    61.73 %
  HOTA  :    26.39 %
  IDs   :       26
  FP    :     1562
  FN    :     3242
  GT    :    16500


### P16: OCSSort | Color | GTALink(Color)

In [33]:
result_16 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p16_yolo_color_ocsort_gtalink(color).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → OCSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 4

Splitting tracklets: 100%|██████████| 150/150 [00:00<00:00, 266.91it/s]


[GTALink] Sau  split: 212 tracklets
[GTALink] Trước merge: 212 tracklets
[GTALink] Sau  merge: 78 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p16_yolo_color_ocsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14850 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    68.85 %
  MOTP  :    65.41 %
  IDF1  :    55.92 %
  HOTA  :    21.74 %
  IDs   :      341
  FP    :     1574
  FN    :     3224
  GT    :    16500


### P17: OCSSort | Color | GTALink(OSNet)

In [34]:
result_17 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p17_yolo_color_ocsort_gtalink(osnet).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → OCSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processi

Splitting tracklets: 100%|██████████| 150/150 [00:00<00:00, 380.73it/s]


[GTALink] Sau  split: 222 tracklets
[GTALink] Trước merge: 222 tracklets
[GTALink] Sau  merge: 36 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p17_yolo_color_ocsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14850 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    68.42 %
  MOTP  :    65.41 %
  IDF1  :    66.19 %
  HOTA  :    26.65 %
  IDs   :      413
  FP    :     1574
  FN    :     3224
  GT    :    16500


### P18: OCSSort | Color | GTALink(Solider)

In [35]:
result_18 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p18_yolo_color_ocsort_gtalink(solider).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → OCSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proces

Splitting tracklets: 100%|██████████| 150/150 [00:00<00:00, 392.45it/s]


[GTALink] Sau  split: 174 tracklets
[GTALink] Trước merge: 174 tracklets
[GTALink] Sau  merge: 32 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p18_yolo_color_ocsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14850 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    70.73 %
  MOTP  :    65.41 %
  IDF1  :    67.94 %
  HOTA  :    28.35 %
  IDs   :       31
  FP    :     1574
  FN    :     3224
  GT    :    16500


### P19: OCSSort | OSNet | GTALink(Color)


In [36]:
result_19 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p19_yolo_osnet_ocsort_gtalink(color).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → OSNET → OCSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processi

Splitting tracklets: 100%|██████████| 134/134 [00:00<00:00, 367.31it/s]


[GTALink] Sau  split: 185 tracklets
[GTALink] Trước merge: 185 tracklets
[GTALink] Sau  merge: 77 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p19_yolo_osnet_ocsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14819 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    68.99 %
  MOTP  :    65.42 %
  IDF1  :    56.24 %
  HOTA  :    22.49 %
  IDs   :      312
  FP    :     1562
  FN    :     3243
  GT    :    16500


### P20: OCSSort | OSNet | GTALink(OSNet)


In [37]:
result_20 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p20_yolo_osnet_ocsort_gtalink(osnet).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → OSNET → OCSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame 4

Splitting tracklets: 100%|██████████| 134/134 [00:00<00:00, 361.93it/s]


[GTALink] Sau  split: 195 tracklets
[GTALink] Trước merge: 195 tracklets
[GTALink] Sau  merge: 35 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p20_yolo_osnet_ocsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14819 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    69.05 %
  MOTP  :    65.42 %
  IDF1  :    64.47 %
  HOTA  :    26.89 %
  IDs   :      301
  FP    :     1562
  FN    :     3243
  GT    :    16500


### P21: OCSSort | OSNet | GTALink(Solider)


In [38]:
result_21 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p21_yolo_osnet_ocsort_gtalink(solider).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → OSNET → OCSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame

Splitting tracklets: 100%|██████████| 134/134 [00:00<00:00, 373.62it/s]


[GTALink] Sau  split: 155 tracklets
[GTALink] Trước merge: 155 tracklets
[GTALink] Sau  merge: 34 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p21_yolo_osnet_ocsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14819 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    70.74 %
  MOTP  :    65.42 %
  IDF1  :    60.88 %
  HOTA  :    27.59 %
  IDs   :       23
  FP    :     1562
  FN    :     3243
  GT    :    16500


### P22: OCSSort | Solider | GTALink(Color)


In [39]:
result_22 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p22_yolo_solider_ocsort_gtalink(color).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_color,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → SOLIDER → OCSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proces

Splitting tracklets: 100%|██████████| 137/137 [00:00<00:00, 323.41it/s]


[GTALink] Sau  split: 189 tracklets
[GTALink] Trước merge: 189 tracklets
[GTALink] Sau  merge: 78 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p22_yolo_solider_ocsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14820 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    68.92 %
  MOTP  :    65.43 %
  IDF1  :    53.98 %
  HOTA  :    22.32 %
  IDs   :      325
  FP    :     1562
  FN    :     3242
  GT    :    16500


### P23: OCSSort | Solider | GTALink(OSNet)


In [40]:
result_23 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p23_yolo_solider_ocsort_gtalink(osnet).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_osnet,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → SOLIDER → OCSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame

Splitting tracklets: 100%|██████████| 137/137 [00:00<00:00, 336.49it/s]


[GTALink] Sau  split: 204 tracklets
[GTALink] Trước merge: 204 tracklets
[GTALink] Sau  merge: 36 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p23_yolo_solider_ocsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14820 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    68.53 %
  MOTP  :    65.43 %
  IDF1  :    64.95 %
  HOTA  :    26.51 %
  IDs   :      389
  FP    :     1562
  FN    :     3242
  GT    :    16500


### P24: OCSSort | Solider | GTALink(Solider)


In [41]:
result_24 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "ocsort",
    os.path.join(OUTPUT_DIR, "p24_yolo_solider_ocsort_gtalink(solider).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_solider,
    tracker_kwargs={"use_emb": True},
)


PIPELINE: YOLO26 → SOLIDER → OCSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fra

Splitting tracklets: 100%|██████████| 137/137 [00:00<00:00, 351.49it/s]


[GTALink] Sau  split: 160 tracklets
[GTALink] Trước merge: 160 tracklets
[GTALink] Sau  merge: 33 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p24_yolo_solider_ocsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14820 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    70.75 %
  MOTP  :    65.43 %
  IDF1  :    62.03 %
  HOTA  :    27.00 %
  IDs   :       23
  FP    :     1562
  FN    :     3242
  GT    :    16500


# DEEPEIOU (9 pipelines)


### P25: DeepEIoU | Color | GTALink(Color)


In [42]:
result_25 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p25_yolo_color_deepeiou_gtalink(color).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_color,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → DEEPEIOU → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame

Splitting tracklets: 100%|██████████| 48/48 [00:00<00:00, 149.87it/s]


[GTALink] Sau  split: 63 tracklets
[GTALink] Trước merge: 63 tracklets
[GTALink] Sau  merge: 42 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p25_yolo_color_deepeiou_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15778 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    66.34 %
  MOTP  :    65.41 %
  IDF1  :    74.42 %
  HOTA  :    26.42 %
  IDs   :      310
  FP    :     2261
  FN    :     2983
  GT    :    16500


### P26: DeepEIoU | Color | GTALink(OSNet)


In [43]:
result_26 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p26_yolo_color_deepeiou_gtalink(osnet).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_osnet,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → DEEPEIOU → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proces

Splitting tracklets: 100%|██████████| 48/48 [00:00<00:00, 169.83it/s]


[GTALink] Sau  split: 64 tracklets
[GTALink] Trước merge: 64 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p26_yolo_color_deepeiou_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15778 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    67.59 %
  MOTP  :    65.41 %
  IDF1  :    76.72 %
  HOTA  :    28.78 %
  IDs   :      103
  FP    :     2261
  FN    :     2983
  GT    :    16500


### P27: DeepEIoU | Color | GTALink(Solider)


In [44]:
result_27 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p27_yolo_color_deepeiou_gtalink(solider).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_solider,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → DEEPEIOU → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proc

Splitting tracklets: 100%|██████████| 48/48 [00:00<00:00, 139.90it/s]


[GTALink] Sau  split: 51 tracklets
[GTALink] Trước merge: 51 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p27_yolo_color_deepeiou_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15778 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    67.67 %
  MOTP  :    65.41 %
  IDF1  :    76.72 %
  HOTA  :    29.04 %
  IDs   :       90
  FP    :     2261
  FN    :     2983
  GT    :    16500


### P28: DeepEIoU | OSNet | GTALink(Color)


In [45]:
result_28 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p28_yolo_osnet_deepeiou_gtalink(color).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_color,
)


PIPELINE: YOLO26 → OSNET → DEEPEIOU → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proces

Splitting tracklets: 100%|██████████| 51/51 [00:00<00:00, 158.96it/s]


[GTALink] Sau  split: 68 tracklets
[GTALink] Trước merge: 68 tracklets
[GTALink] Sau  merge: 44 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p28_yolo_osnet_deepeiou_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15845 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    66.61 %
  MOTP  :    65.42 %
  IDF1  :    71.20 %
  HOTA  :    25.69 %
  IDs   :      287
  FP    :     2284
  FN    :     2939
  GT    :    16500


### P29: DeepEIoU | OSNet | GTALink(OSNet)


In [46]:
result_29 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p29_yolo_osnet_deepeiou_gtalink(osnet).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_osnet,
)


PIPELINE: YOLO26 → OSNET → DEEPEIOU → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing frame

Splitting tracklets: 100%|██████████| 51/51 [00:00<00:00, 148.84it/s]


[GTALink] Sau  split: 66 tracklets
[GTALink] Trước merge: 66 tracklets
[GTALink] Sau  merge: 26 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p29_yolo_osnet_deepeiou_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15845 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    67.75 %
  MOTP  :    65.42 %
  IDF1  :    71.55 %
  HOTA  :    28.44 %
  IDs   :       98
  FP    :     2284
  FN    :     2939
  GT    :    16500


### P30: DeepEIoU | OSNet | GTALink(Solider)


In [47]:
result_30 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p30_yolo_osnet_deepeiou_gtalink(solider).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_solider,
)


PIPELINE: YOLO26 → OSNET → DEEPEIOU → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fra

Splitting tracklets: 100%|██████████| 51/51 [00:00<00:00, 150.44it/s]


[GTALink] Sau  split: 55 tracklets
[GTALink] Trước merge: 55 tracklets
[GTALink] Sau  merge: 28 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p30_yolo_osnet_deepeiou_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15845 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    67.75 %
  MOTP  :    65.42 %
  IDF1  :    72.21 %
  HOTA  :    29.12 %
  IDs   :       99
  FP    :     2284
  FN    :     2939
  GT    :    16500


### P31: DeepEIoU | Solider | GTALink(Color)


In [48]:
result_31 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p31_yolo_solider_deepeiou_gtalink(color).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_color,
)


PIPELINE: YOLO26 → SOLIDER → DEEPEIOU → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proc

Splitting tracklets: 100%|██████████| 57/57 [00:00<00:00, 174.61it/s]


[GTALink] Sau  split: 71 tracklets
[GTALink] Trước merge: 71 tracklets
[GTALink] Sau  merge: 41 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p31_yolo_solider_deepeiou_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15871 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    66.30 %
  MOTP  :    65.41 %
  IDF1  :    78.69 %
  HOTA  :    27.43 %
  IDs   :      304
  FP    :     2314
  FN    :     2943
  GT    :    16500


### P32: DeepEIoU | Solider | GTALink(OSNet)


In [49]:
result_32 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p32_yolo_solider_deepeiou_gtalink(osnet).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_osnet,
)


PIPELINE: YOLO26 → SOLIDER → DEEPEIOU → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fra

Splitting tracklets: 100%|██████████| 57/57 [00:00<00:00, 188.62it/s]


[GTALink] Sau  split: 74 tracklets
[GTALink] Trước merge: 74 tracklets
[GTALink] Sau  merge: 26 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p32_yolo_solider_deepeiou_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15871 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    67.44 %
  MOTP  :    65.41 %
  IDF1  :    80.60 %
  HOTA  :    29.44 %
  IDs   :      115
  FP    :     2314
  FN    :     2943
  GT    :    16500


### P33: DeepEIoU | Solider | GTALink(Solider)


In [50]:
result_33 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "deepeiou",
    os.path.join(OUTPUT_DIR, "p33_yolo_solider_deepeiou_gtalink(solider).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_solider,
)


PIPELINE: YOLO26 → SOLIDER → DEEPEIOU → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing f

Splitting tracklets: 100%|██████████| 57/57 [00:00<00:00, 157.66it/s]


[GTALink] Sau  split: 62 tracklets
[GTALink] Trước merge: 62 tracklets
[GTALink] Sau  merge: 28 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p33_yolo_solider_deepeiou_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15871 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    67.53 %
  MOTP  :    65.41 %
  IDF1  :    79.02 %
  HOTA  :    28.29 %
  IDs   :      101
  FP    :     2314
  FN    :     2943
  GT    :    16500


# STRONGSORT (9 pipelines)


### P34: StrongSORT | Color | GTALink(Color)


In [51]:
result_34 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p34_yolo_color_strongsort_gtalink(color).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_color,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → STRONGSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing fra

Splitting tracklets: 100%|██████████| 85/85 [00:00<00:00, 309.94it/s]


[GTALink] Sau  split: 103 tracklets
[GTALink] Trước merge: 103 tracklets
[GTALink] Sau  merge: 42 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p34_yolo_color_strongsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13144 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    62.97 %
  MOTP  :    65.32 %
  IDF1  :    63.25 %
  HOTA  :    25.71 %
  IDs   :      172
  FP    :     1291
  FN    :     4647
  GT    :    16500


### P35: StrongSORT | Color | GTALink(OSNet)


In [52]:
result_35 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p35_yolo_color_strongsort_gtalink(osnet).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_osnet,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → STRONGSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proc

Splitting tracklets: 100%|██████████| 85/85 [00:00<00:00, 331.08it/s]


[GTALink] Sau  split: 104 tracklets
[GTALink] Trước merge: 104 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p35_yolo_color_strongsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13144 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    63.56 %
  MOTP  :    65.32 %
  IDF1  :    65.88 %
  HOTA  :    29.07 %
  IDs   :       74
  FP    :     1291
  FN    :     4647
  GT    :    16500


### P36: StrongSORT | Color | GTALink(Solider)


In [53]:
result_36 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p36_yolo_color_strongsort_gtalink(solider).avi"),
    extractor=extractor_color, refiner=GTALink(), refiner_extractor=extractor_solider,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → STRONGSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Pr

Splitting tracklets: 100%|██████████| 85/85 [00:00<00:00, 295.83it/s]


[GTALink] Sau  split: 92 tracklets
[GTALink] Trước merge: 92 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p36_yolo_color_strongsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13144 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    63.98 %
  MOTP  :    65.32 %
  IDF1  :    68.41 %
  HOTA  :    30.46 %
  IDs   :        5
  FP    :     1291
  FN    :     4647
  GT    :    16500


### P37: StrongSORT | OSNet | GTALink(Color)


In [54]:
result_37 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p37_yolo_osnet_strongsort_gtalink(color).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_color,
)


PIPELINE: YOLO26 → OSNET → STRONGSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Proc

Splitting tracklets: 100%|██████████| 84/84 [00:00<00:00, 166.17it/s]


[GTALink] Sau  split: 104 tracklets
[GTALink] Trước merge: 104 tracklets
[GTALink] Sau  merge: 44 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p37_yolo_osnet_strongsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13156 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    62.86 %
  MOTP  :    65.32 %
  IDF1  :    63.26 %
  HOTA  :    25.28 %
  IDs   :      196
  FP    :     1294
  FN    :     4638
  GT    :    16500


### P38: StrongSORT | OSNet | GTALink(OSNet)


In [55]:
result_38 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p38_yolo_osnet_strongsort_gtalink(osnet).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_osnet,
)


PIPELINE: YOLO26 → OSNET → STRONGSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing fra

Splitting tracklets: 100%|██████████| 84/84 [00:00<00:00, 181.04it/s]


[GTALink] Sau  split: 104 tracklets
[GTALink] Trước merge: 104 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p38_yolo_osnet_strongsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13156 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    63.63 %
  MOTP  :    65.32 %
  IDF1  :    66.87 %
  HOTA  :    29.65 %
  IDs   :       69
  FP    :     1294
  FN    :     4638
  GT    :    16500


### P39: StrongSORT | OSNet | GTALink(Solider)


In [56]:
result_39 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p39_yolo_osnet_strongsort_gtalink(solider).avi"),
    extractor=extractor_osnet, refiner=GTALink(), refiner_extractor=extractor_solider,
)


PIPELINE: YOLO26 → OSNET → STRONGSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing f

Splitting tracklets: 100%|██████████| 84/84 [00:00<00:00, 158.75it/s]


[GTALink] Sau  split: 91 tracklets
[GTALink] Trước merge: 91 tracklets
[GTALink] Sau  merge: 28 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p39_yolo_osnet_strongsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13156 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    64.02 %
  MOTP  :    65.32 %
  IDF1  :    68.83 %
  HOTA  :    31.51 %
  IDs   :        5
  FP    :     1294
  FN    :     4638
  GT    :    16500


### P40: StrongSORT | Solider | GTALink(Color)


In [57]:
result_40 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p40_yolo_solider_strongsort_gtalink(color).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_color,
)


PIPELINE: YOLO26 → SOLIDER → STRONGSORT → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Pr

Splitting tracklets: 100%|██████████| 85/85 [00:00<00:00, 150.83it/s]


[GTALink] Sau  split: 103 tracklets
[GTALink] Trước merge: 103 tracklets
[GTALink] Sau  merge: 42 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p40_yolo_solider_strongsort_gtalink(color).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13149 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    63.00 %
  MOTP  :    65.32 %
  IDF1  :    63.24 %
  HOTA  :    25.71 %
  IDs   :      172
  FP    :     1291
  FN    :     4642
  GT    :    16500


### P41: StrongSORT | Solider | GTALink(OSNet)


In [58]:
result_41 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p41_yolo_solider_strongsort_gtalink(osnet).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_osnet,
)


PIPELINE: YOLO26 → SOLIDER → STRONGSORT → GTALink(osnet)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing f

Splitting tracklets: 100%|██████████| 85/85 [00:00<00:00, 185.03it/s]


[GTALink] Sau  split: 104 tracklets
[GTALink] Trước merge: 104 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p41_yolo_solider_strongsort_gtalink(osnet).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13149 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    63.59 %
  MOTP  :    65.32 %
  IDF1  :    65.88 %
  HOTA  :    29.07 %
  IDs   :       74
  FP    :     1291
  FN    :     4642
  GT    :    16500


### P42: StrongSORT | Solider | GTALink(Solider)


In [59]:
result_42 = run_pipeline(
    VIDEO_PATH, GT_CSV_PATH, detector, "strongsort",
    os.path.join(OUTPUT_DIR, "p42_yolo_solider_strongsort_gtalink(solider).avi"),
    extractor=extractor_solider, refiner=GTALink(), refiner_extractor=extractor_solider,
)


PIPELINE: YOLO26 → SOLIDER → STRONGSORT → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Processing

Splitting tracklets: 100%|██████████| 85/85 [00:00<00:00, 148.00it/s]


[GTALink] Sau  split: 92 tracklets
[GTALink] Trước merge: 92 tracklets
[GTALink] Sau  merge: 27 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\p42_yolo_solider_strongsort_gtalink(solider).avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 13149 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    64.01 %
  MOTP  :    65.32 %
  IDF1  :    68.42 %
  HOTA  :    30.47 %
  IDs   :        5
  FP    :     1291
  FN    :     4642
  GT    :    16500


# SO SÁNH KẾT QUẢ

In [60]:
all_results = {
    # ByteTrack
    "P01 ByteTrack | None    | GTA(Color)"   : result_1['metrics'],
    "P02 ByteTrack | None    | GTA(OSNet)"   : result_2['metrics'],
    "P03 ByteTrack | None    | GTA(Solider)" : result_3['metrics'],
    "P04 ByteTrack | Color   | GTA(Color)"   : result_4['metrics'],
    "P05 ByteTrack | Color   | GTA(OSNet)"   : result_5['metrics'],
    "P06 ByteTrack | Color   | GTA(Solider)" : result_6['metrics'],
    "P07 ByteTrack | OSNet   | GTA(Color)"   : result_7['metrics'],
    "P08 ByteTrack | OSNet   | GTA(OSNet)"   : result_8['metrics'],
    "P09 ByteTrack | OSNet   | GTA(Solider)" : result_9['metrics'],
    "P10 ByteTrack | Solider | GTA(Color)"   : result_10['metrics'],
    "P11 ByteTrack | Solider | GTA(OSNet)"   : result_11['metrics'],
    "P12 ByteTrack | Solider | GTA(Solider)" : result_12['metrics'],
    # OCSSort
    "P13 OCSSort   | None    | GTA(Color)"   : result_13['metrics'],
    "P14 OCSSort   | None    | GTA(OSNet)"   : result_14['metrics'],
    "P15 OCSSort   | None    | GTA(Solider)" : result_15['metrics'],
    "P16 OCSSort   | Color   | GTA(Color)"   : result_16['metrics'],
    "P17 OCSSort   | Color   | GTA(OSNet)"   : result_17['metrics'],
    "P18 OCSSort   | Color   | GTA(Solider)" : result_18['metrics'],
    "P19 OCSSort   | OSNet   | GTA(Color)"   : result_19['metrics'],
    "P20 OCSSort   | OSNet   | GTA(OSNet)"   : result_20['metrics'],
    "P21 OCSSort   | OSNet   | GTA(Solider)" : result_21['metrics'],
    "P22 OCSSort   | Solider | GTA(Color)"   : result_22['metrics'],
    "P23 OCSSort   | Solider | GTA(OSNet)"   : result_23['metrics'],
    "P24 OCSSort   | Solider | GTA(Solider)" : result_24['metrics'],
    # DeepEIoU
    "P25 DeepEIoU  | Color   | GTA(Color)"   : result_25['metrics'],
    "P26 DeepEIoU  | Color   | GTA(OSNet)"   : result_26['metrics'],
    "P27 DeepEIoU  | Color   | GTA(Solider)" : result_27['metrics'],
    "P28 DeepEIoU  | OSNet   | GTA(Color)"   : result_28['metrics'],
    "P29 DeepEIoU  | OSNet   | GTA(OSNet)"   : result_29['metrics'],
    "P30 DeepEIoU  | OSNet   | GTA(Solider)" : result_30['metrics'],
    "P31 DeepEIoU  | Solider | GTA(Color)"   : result_31['metrics'],
    "P32 DeepEIoU  | Solider | GTA(OSNet)"   : result_32['metrics'],
    "P33 DeepEIoU  | Solider | GTA(Solider)" : result_33['metrics'],
    # StrongSORT
    "P34 StrongSORT| Color   | GTA(Color)"   : result_34['metrics'],
    "P35 StrongSORT| Color   | GTA(OSNet)"   : result_35['metrics'],
    "P36 StrongSORT| Color   | GTA(Solider)" : result_36['metrics'],
    "P37 StrongSORT| OSNet   | GTA(Color)"   : result_37['metrics'],
    "P38 StrongSORT| OSNet   | GTA(OSNet)"   : result_38['metrics'],
    "P39 StrongSORT| OSNet   | GTA(Solider)" : result_39['metrics'],
    "P40 StrongSORT| Solider | GTA(Color)"   : result_40['metrics'],
    "P41 StrongSORT| Solider | GTA(OSNet)"   : result_41['metrics'],
    "P42 StrongSORT| Solider | GTA(Solider)" : result_42['metrics'],
}
 
print("\n" + "="*95)
print(f"{'Pipeline':<42} {'MOTA':>6} {'MOTP':>6} {'IDF1':>6} {'HOTA':>6} {'IDs':>5} {'FP':>6} {'FN':>6}")
print("="*95)
 
prev_tracker = ""
for name, m in all_results.items():
    # In dòng phân cách giữa các nhóm tracker
    cur_tracker = name.split()[1]
    if cur_tracker != prev_tracker:
        if prev_tracker:
            print("-" * 95)
        prev_tracker = cur_tracker
 
    print(f"{name:<42} {m['MOTA']:>6.1f} {m['MOTP']:>6.1f} "
          f"{m['IDF1']:>6.1f} {m['HOTA']:>6.1f} {m['IDs']:>5d} "
          f"{m['FP']:>6d} {m['FN']:>6d}")
 
print("="*95)


Pipeline                                     MOTA   MOTP   IDF1   HOTA   IDs     FP     FN
P01 ByteTrack | None    | GTA(Color)         72.8   65.4   68.8   29.7   131   1443   2917
P02 ByteTrack | None    | GTA(OSNet)         73.2   65.4   73.8   36.9    55   1443   2917
P03 ByteTrack | None    | GTA(Solider)       73.5   65.4   71.3   34.1     6   1443   2917
P04 ByteTrack | Color   | GTA(Color)         73.0   65.4   69.9   30.5    68   1437   2952
P05 ByteTrack | Color   | GTA(OSNet)         73.0   65.4   72.1   36.0    75   1437   2952
P06 ByteTrack | Color   | GTA(Solider)       73.4   65.4   73.0   36.2     1   1437   2952
P07 ByteTrack | OSNet   | GTA(Color)         73.1   65.4   70.4   30.6    68   1435   2931
P08 ByteTrack | OSNet   | GTA(OSNet)         73.1   65.4   72.9   34.5    75   1435   2931
P09 ByteTrack | OSNet   | GTA(Solider)       73.5   65.4   72.3   35.5     1   1435   2931
P10 ByteTrack | Solider | GTA(Color)         73.1   65.4   69.7   30.4    68   1433   294

In [61]:
# In top 5 theo MOTA
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['MOTA'], reverse=True)
print("\nTop 5 pipelines theo MOTA:")
for name, m in sorted_results[:5]:
    print(f"  {name:<42} MOTA={m['MOTA']:6.1f}  IDF1={m['IDF1']:6.1f}  HOTA={m['HOTA']:6.1f}")


Top 5 pipelines theo MOTA:
  P03 ByteTrack | None    | GTA(Solider)     MOTA=  73.5  IDF1=  71.3  HOTA=  34.1
  P09 ByteTrack | OSNet   | GTA(Solider)     MOTA=  73.5  IDF1=  72.3  HOTA=  35.5
  P12 ByteTrack | Solider | GTA(Solider)     MOTA=  73.5  IDF1=  72.8  HOTA=  35.0
  P06 ByteTrack | Color   | GTA(Solider)     MOTA=  73.4  IDF1=  73.0  HOTA=  36.2
  P02 ByteTrack | None    | GTA(OSNet)       MOTA=  73.2  IDF1=  73.8  HOTA=  36.9
